# Exploratory Survival Analysis - Week 3

## Comprehensive Survival Analysis on Production Dataset

**Session:** Week 3 (March Week 2)  
**Duration:** 25-30 hours (survival analysis portion)  
**Objective:** Perform comprehensive univariate and multivariate survival analysis

**What we'll analyze:**
1. **Univariate survival analysis** - Each clinical feature vs survival
2. **Extended Kaplan-Meier curves** - All subtypes, risk groups, treatments
3. **Log-rank tests** - Statistical comparisons between groups
4. **Univariate Cox models** - Hazard ratios for each feature
5. **Pathway-survival associations** - Which pathways predict survival
6. **Multivariate Cox models** - Combined feature effects

**Input:** Production splits (train: 1,995 patients)  
**Output:** Publication-ready survival analysis + hazard ratio tables

Let's uncover the survival patterns! 📊

In [8]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 11
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'

# Paths
project_dir = Path(r'D:\Projects\tcga-metabric-treatment-ai')
data_dir = project_dir / 'data' / 'merged' / 'final_splits'
results_dir = project_dir / 'results'
figures_dir = results_dir / 'figures' / 'survival_analysis'
tables_dir = results_dir / 'tables' / 'survival_analysis'
figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

# Load production training data
print("="*70)
print("WEEK 3: EXPLORATORY SURVIVAL ANALYSIS")
print("="*70)

print("\nLoading production training data...")
train = pd.read_csv(data_dir / 'train_final.csv')

print(f"\nTraining set loaded: {train.shape}")
print(f"  Patients: {train.shape[0]}")
print(f"  Features: {train.shape[1]}")

# Check survival data availability
os_complete = train['os_days'].notna() & train['os_status'].notna()
rfs_complete = train['rfs_days'].notna() & train['rfs_status'].notna()

print(f"\nSurvival data availability:")
print(f"  Overall Survival (OS): {os_complete.sum()} / {len(train)} ({os_complete.sum()/len(train)*100:.1f}%)")
print(f"  Recurrence-Free Survival (RFS): {rfs_complete.sum()} / {len(train)} ({rfs_complete.sum()/len(train)*100:.1f}%)")

# Basic survival statistics
print(f"\nOverall Survival (OS) statistics:")
print(f"  Median follow-up: {train['os_days'].median():.0f} days ({train['os_days'].median()/365.25:.1f} years)")
print(f"  Max follow-up: {train['os_days'].max():.0f} days ({train['os_days'].max()/365.25:.1f} years)")
print(f"  Events (deaths): {(train['os_status'] == 1).sum()} ({(train['os_status'] == 1).sum()/len(train)*100:.1f}%)")
print(f"  Censored (alive): {(train['os_status'] == 0).sum()} ({(train['os_status'] == 0).sum()/len(train)*100:.1f}%)")

print("\n✅ Data loaded successfully!")
print("   Ready for survival analysis")

WEEK 3: EXPLORATORY SURVIVAL ANALYSIS

Loading production training data...

Training set loaded: (1995, 110)
  Patients: 1995
  Features: 110

Survival data availability:
  Overall Survival (OS): 1995 / 1995 (100.0%)
  Recurrence-Free Survival (RFS): 1237 / 1995 (62.0%)

Overall Survival (OS) statistics:
  Median follow-up: 2181 days (6.0 years)
  Max follow-up: 10812 days (29.6 years)
  Events (deaths): 831 (41.7%)
  Censored (alive): 1164 (58.3%)

✅ Data loaded successfully!
   Ready for survival analysis


### Part 1: Univariate Kaplan-Meier Survival Analysis

**Objective:** Analyze survival by key clinical features

**Features to analyze:**
1. Molecular subtype (Hormone+, Triple-, HER2+)
2. PAM50 subtype (LumA, LumB, Basal, Her2, Normal)
3. Risk group (Low, Intermediate, High)
4. Cohort (TCGA vs METABRIC)
5. ER status
6. Stage (0-4)

For each: KM curves + log-rank test + median survival

In [9]:
# Part 1: Univariate Kaplan-Meier Analysis
print("="*70)
print("PART 1: UNIVARIATE KAPLAN-MEIER ANALYSIS")
print("="*70)

# Initialize Kaplan-Meier fitter
kmf = KaplanMeierFitter()

# Storage for results
survival_results = []

# Figure 1: Survival by Molecular Subtype
print("\n1. SURVIVAL BY MOLECULAR SUBTYPE")
print("="*70)

fig, ax = plt.subplots(figsize=(12, 7))

subtype_colors = {
    'Hormone_Positive': '#2ECC71',
    'Triple_Negative': '#E74C3C',
    'HER2_Positive': '#9B59B6'
}

subtypes_to_plot = ['Hormone_Positive', 'Triple_Negative', 'HER2_Positive']
subtype_data = {}

for subtype in subtypes_to_plot:
    mask = train['molecular_subtype'] == subtype
    if mask.sum() > 10:
        time = train.loc[mask, 'os_days']
        event = train.loc[mask, 'os_status']
        
        # Fit KM
        kmf.fit(time, event, label=subtype.replace('_', ' '))
        kmf.plot_survival_function(ax=ax, color=subtype_colors[subtype], linewidth=2.5)
        
        # Store data for log-rank test
        subtype_data[subtype] = (time, event)
        
        # Calculate median survival
        median_survival = kmf.median_survival_time_
        n_patients = mask.sum()
        n_events = event.sum()
        
        survival_results.append({
            'Variable': 'Molecular Subtype',
            'Group': subtype.replace('_', ' '),
            'N': n_patients,
            'Events': n_events,
            'Median_Survival_Days': median_survival,
            'Median_Survival_Years': median_survival / 365.25 if pd.notna(median_survival) else np.nan
        })
        
        print(f"\n{subtype.replace('_', ' ')}:")
        print(f"  N = {n_patients}")
        print(f"  Events = {n_events} ({n_events/n_patients*100:.1f}%)")
        if pd.notna(median_survival):
            print(f"  Median survival = {median_survival:.0f} days ({median_survival/365.25:.1f} years)")
        else:
            print(f"  Median survival = Not reached")

# Log-rank test
if len(subtype_data) >= 2:
    groups = list(subtype_data.keys())
    T1, E1 = subtype_data[groups[0]]
    T2, E2 = subtype_data[groups[1]]
    
    lr_result = logrank_test(T1, T2, E1, E2)
    print(f"\nLog-rank test (all subtypes):")
    print(f"  Test statistic = {lr_result.test_statistic:.3f}")
    print(f"  p-value = {lr_result.p_value:.4f}")
    
    if lr_result.p_value < 0.001:
        print(f"  Result: *** Highly significant difference (p < 0.001)")
    elif lr_result.p_value < 0.01:
        print(f"  Result: ** Significant difference (p < 0.01)")
    elif lr_result.p_value < 0.05:
        print(f"  Result: * Significant difference (p < 0.05)")
    else:
        print(f"  Result: No significant difference (p >= 0.05)")

ax.set_xlabel('Time (years)', fontsize=12)
ax.set_ylabel('Overall Survival Probability', fontsize=12)
ax.set_title('Kaplan-Meier Curves by Molecular Subtype', fontsize=14, fontweight='bold')
ax.set_xlim(0, train['os_days'].max() / 365.25)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11, loc='lower left')
ax.grid(alpha=0.3)

# Convert x-axis to years
current_ticks = ax.get_xticks()
ax.set_xticklabels([f'{int(x)}' for x in current_ticks])

plt.tight_layout()
mol_subtype_path = figures_dir / 'km_molecular_subtype.png'
plt.savefig(mol_subtype_path)
print(f"\n✅ Saved: {mol_subtype_path}")
plt.close()

# Figure 2: Survival by PAM50 Subtype
print("\n" + "="*70)
print("2. SURVIVAL BY PAM50 SUBTYPE")
print("="*70)

fig, ax = plt.subplots(figsize=(12, 7))

pam50_colors = {
    'LumA': '#2ECC71',
    'LumB': '#F39C12',
    'Basal': '#E74C3C',
    'Her2': '#9B59B6',
    'Normal': '#95A5A6'
}

for pam50 in ['LumA', 'LumB', 'Basal', 'Her2', 'Normal']:
    mask = train['pam50_subtype'] == pam50
    if mask.sum() > 10:
        time = train.loc[mask, 'os_days']
        event = train.loc[mask, 'os_status']
        
        kmf.fit(time, event, label=pam50)
        kmf.plot_survival_function(ax=ax, color=pam50_colors[pam50], linewidth=2.5)
        
        median_survival = kmf.median_survival_time_
        n_patients = mask.sum()
        n_events = event.sum()
        
        survival_results.append({
            'Variable': 'PAM50 Subtype',
            'Group': pam50,
            'N': n_patients,
            'Events': n_events,
            'Median_Survival_Days': median_survival,
            'Median_Survival_Years': median_survival / 365.25 if pd.notna(median_survival) else np.nan
        })
        
        print(f"\n{pam50}:")
        print(f"  N = {n_patients}")
        print(f"  Events = {n_events} ({n_events/n_patients*100:.1f}%)")
        if pd.notna(median_survival):
            print(f"  Median survival = {median_survival:.0f} days ({median_survival/365.25:.1f} years)")
        else:
            print(f"  Median survival = Not reached")

ax.set_xlabel('Time (years)', fontsize=12)
ax.set_ylabel('Overall Survival Probability', fontsize=12)
ax.set_title('Kaplan-Meier Curves by PAM50 Subtype', fontsize=14, fontweight='bold')
ax.set_xlim(0, train['os_days'].max() / 365.25)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11, loc='lower left')
ax.grid(alpha=0.3)

current_ticks = ax.get_xticks()
ax.set_xticklabels([f'{int(x)}' for x in current_ticks])

plt.tight_layout()
pam50_path = figures_dir / 'km_pam50_subtype.png'
plt.savefig(pam50_path)
print(f"\n✅ Saved: {pam50_path}")
plt.close()

print("\n" + "="*70)
print("UNIVARIATE KM ANALYSIS COMPLETE (Part 1/3)")
print("="*70)
print(f"\n✅ Created 2 Kaplan-Meier figures")
print(f"   Analyzed {len(survival_results)} groups")

PART 1: UNIVARIATE KAPLAN-MEIER ANALYSIS

1. SURVIVAL BY MOLECULAR SUBTYPE

Hormone Positive:
  N = 1247
  Events = 565.0 (45.3%)
  Median survival = 5010 days (13.7 years)

Triple Negative:
  N = 217
  Events = 96.0 (44.2%)
  Median survival = 4173 days (11.4 years)

HER2 Positive:
  N = 286
  Events = 118.0 (41.3%)
  Median survival = 3070 days (8.4 years)

Log-rank test (all subtypes):
  Test statistic = 8.175
  p-value = 0.0042
  Result: ** Significant difference (p < 0.01)

✅ Saved: D:\Projects\tcga-metabric-treatment-ai\results\figures\survival_analysis\km_molecular_subtype.png

2. SURVIVAL BY PAM50 SUBTYPE

LumA:
  N = 771
  Events = 289.0 (37.5%)
  Median survival = 5681 days (15.6 years)

LumB:
  N = 594
  Events = 259.0 (43.6%)
  Median survival = 3714 days (10.2 years)

Basal:
  N = 281
  Events = 102.0 (36.3%)
  Median survival = 5558 days (15.2 years)

Her2:
  N = 232
  Events = 125.0 (53.9%)
  Median survival = 3244 days (8.9 years)

Normal:
  N = 117
  Events = 56.0 (47.

### Part 2: Additional Kaplan-Meier Analyses

**Continuing univariate analysis:**
- Risk group (Low/Intermediate/High)
- Stage (0-4)
- ER status
- Treatment effects

In [10]:
# Part 2: Additional KM Analyses
print("="*70)
print("PART 2: ADDITIONAL KAPLAN-MEIER ANALYSES")
print("="*70)

# Figure 3: Survival by Risk Group
print("\n3. SURVIVAL BY RISK GROUP")
print("="*70)

fig, ax = plt.subplots(figsize=(12, 7))

risk_colors = {
    'Low_Risk': '#2ECC71',
    'Intermediate_Risk': '#F39C12',
    'High_Risk': '#E74C3C'
}

for risk in ['Low_Risk', 'Intermediate_Risk', 'High_Risk']:
    mask = train['risk_group'] == risk
    if mask.sum() > 10:
        time = train.loc[mask, 'os_days']
        event = train.loc[mask, 'os_status']
        
        kmf.fit(time, event, label=risk.replace('_', ' '))
        kmf.plot_survival_function(ax=ax, color=risk_colors[risk], linewidth=2.5)
        
        median_survival = kmf.median_survival_time_
        n_patients = mask.sum()
        n_events = event.sum()
        
        survival_results.append({
            'Variable': 'Risk Group',
            'Group': risk.replace('_', ' '),
            'N': n_patients,
            'Events': n_events,
            'Median_Survival_Days': median_survival,
            'Median_Survival_Years': median_survival / 365.25 if pd.notna(median_survival) else np.nan
        })
        
        print(f"\n{risk.replace('_', ' ')}:")
        print(f"  N = {n_patients}")
        print(f"  Events = {n_events} ({n_events/n_patients*100:.1f}%)")
        if pd.notna(median_survival):
            print(f"  Median survival = {median_survival:.0f} days ({median_survival/365.25:.1f} years)")
        else:
            print(f"  Median survival = Not reached")

ax.set_xlabel('Time (years)', fontsize=12)
ax.set_ylabel('Overall Survival Probability', fontsize=12)
ax.set_title('Kaplan-Meier Curves by Risk Group', fontsize=14, fontweight='bold')
ax.set_xlim(0, train['os_days'].max() / 365.25)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11, loc='lower left')
ax.grid(alpha=0.3)

current_ticks = ax.get_xticks()
ax.set_xticklabels([f'{int(x)}' for x in current_ticks])

plt.tight_layout()
risk_path = figures_dir / 'km_risk_group.png'
plt.savefig(risk_path)
print(f"\n✅ Saved: {risk_path}")
plt.close()

# Figure 4: Survival by Stage
print("\n" + "="*70)
print("4. SURVIVAL BY STAGE")
print("="*70)

fig, ax = plt.subplots(figsize=(12, 7))

stage_colors = {0: '#2ECC71', 1: '#3498DB', 2: '#F39C12', 3: '#E67E22', 4: '#E74C3C'}

for stage in [0, 1, 2, 3, 4]:
    mask = train['stage_imputed'] == stage
    if mask.sum() > 10:
        time = train.loc[mask, 'os_days']
        event = train.loc[mask, 'os_status']
        
        kmf.fit(time, event, label=f'Stage {stage}')
        kmf.plot_survival_function(ax=ax, color=stage_colors[stage], linewidth=2.5)
        
        median_survival = kmf.median_survival_time_
        n_patients = mask.sum()
        n_events = event.sum()
        
        survival_results.append({
            'Variable': 'Stage',
            'Group': f'Stage {stage}',
            'N': n_patients,
            'Events': n_events,
            'Median_Survival_Days': median_survival,
            'Median_Survival_Years': median_survival / 365.25 if pd.notna(median_survival) else np.nan
        })
        
        print(f"\nStage {stage}:")
        print(f"  N = {n_patients}")
        print(f"  Events = {n_events} ({n_events/n_patients*100:.1f}%)")
        if pd.notna(median_survival):
            print(f"  Median survival = {median_survival:.0f} days ({median_survival/365.25:.1f} years)")
        else:
            print(f"  Median survival = Not reached")

ax.set_xlabel('Time (years)', fontsize=12)
ax.set_ylabel('Overall Survival Probability', fontsize=12)
ax.set_title('Kaplan-Meier Curves by Stage', fontsize=14, fontweight='bold')
ax.set_xlim(0, train['os_days'].max() / 365.25)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11, loc='lower left')
ax.grid(alpha=0.3)

current_ticks = ax.get_xticks()
ax.set_xticklabels([f'{int(x)}' for x in current_ticks])

plt.tight_layout()
stage_path = figures_dir / 'km_stage.png'
plt.savefig(stage_path)
print(f"\n✅ Saved: {stage_path}")
plt.close()

# Figure 5: Survival by ER Status
print("\n" + "="*70)
print("5. SURVIVAL BY ER STATUS")
print("="*70)

fig, ax = plt.subplots(figsize=(12, 7))

er_colors = {'Positive': '#2ECC71', 'Negative': '#E74C3C'}

for er in ['Positive', 'Negative']:
    mask = train['er_status'] == er
    if mask.sum() > 10:
        time = train.loc[mask, 'os_days']
        event = train.loc[mask, 'os_status']
        
        kmf.fit(time, event, label=f'ER {er}')
        kmf.plot_survival_function(ax=ax, color=er_colors[er], linewidth=2.5)
        
        median_survival = kmf.median_survival_time_
        n_patients = mask.sum()
        n_events = event.sum()
        
        survival_results.append({
            'Variable': 'ER Status',
            'Group': f'ER {er}',
            'N': n_patients,
            'Events': n_events,
            'Median_Survival_Days': median_survival,
            'Median_Survival_Years': median_survival / 365.25 if pd.notna(median_survival) else np.nan
        })
        
        print(f"\nER {er}:")
        print(f"  N = {n_patients}")
        print(f"  Events = {n_events} ({n_events/n_patients*100:.1f}%)")
        if pd.notna(median_survival):
            print(f"  Median survival = {median_survival:.0f} days ({median_survival/365.25:.1f} years)")
        else:
            print(f"  Median survival = Not reached")

ax.set_xlabel('Time (years)', fontsize=12)
ax.set_ylabel('Overall Survival Probability', fontsize=12)
ax.set_title('Kaplan-Meier Curves by ER Status', fontsize=14, fontweight='bold')
ax.set_xlim(0, train['os_days'].max() / 365.25)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11, loc='best')
ax.grid(alpha=0.3)

current_ticks = ax.get_xticks()
ax.set_xticklabels([f'{int(x)}' for x in current_ticks])

plt.tight_layout()
er_path = figures_dir / 'km_er_status.png'
plt.savefig(er_path)
print(f"\n✅ Saved: {er_path}")
plt.close()

# Save survival results table
print("\n" + "="*70)
print("SAVING SURVIVAL SUMMARY TABLE")
print("="*70)

survival_df = pd.DataFrame(survival_results)
survival_table_path = tables_dir / 'univariate_survival_summary.csv'
survival_df.to_csv(survival_table_path, index=False)

print(f"\n✅ Saved: {survival_table_path}")
print(f"\nSummary table preview:")
print(survival_df.head(10).to_string(index=False))

print("\n" + "="*70)
print("PART 2 COMPLETE")
print("="*70)
print(f"\n✅ Total KM figures created: 5")
print(f"   Total groups analyzed: {len(survival_results)}")

PART 2: ADDITIONAL KAPLAN-MEIER ANALYSES

3. SURVIVAL BY RISK GROUP

Low Risk:
  N = 42
  Events = 17.0 (40.5%)
  Median survival = 7307 days (20.0 years)

Intermediate Risk:
  N = 1181
  Events = 398.0 (33.7%)
  Median survival = 5214 days (14.3 years)

High Risk:
  N = 772
  Events = 416.0 (53.9%)
  Median survival = 3649 days (10.0 years)

✅ Saved: D:\Projects\tcga-metabric-treatment-ai\results\figures\survival_analysis\km_risk_group.png

4. SURVIVAL BY STAGE

Stage 1:
  N = 483
  Events = 180.0 (37.3%)
  Median survival = 6459 days (17.7 years)

Stage 2:
  N = 1240
  Events = 549.0 (44.3%)
  Median survival = 4161 days (11.4 years)

Stage 3:
  N = 245
  Events = 87.0 (35.5%)
  Median survival = 3582 days (9.8 years)

Stage 4:
  N = 18
  Events = 14.0 (77.8%)
  Median survival = 1152 days (3.2 years)

✅ Saved: D:\Projects\tcga-metabric-treatment-ai\results\figures\survival_analysis\km_stage.png

5. SURVIVAL BY ER STATUS

ER Positive:
  N = 1559
  Events = 651.0 (41.8%)
  Median surv

### Part 3: Univariate Cox Proportional Hazards Analysis

**Objective:** Calculate hazard ratios for each clinical feature

**What we'll compute:**
- Hazard ratios (HR) with 95% confidence intervals
- P-values for statistical significance
- Concordance index (C-index) for each model

**Features to analyze:**
- Age (continuous)
- Stage (ordinal: 0-4)
- Lymph nodes (continuous)
- Molecular subtype (categorical)
- PAM50 subtype (categorical)
- Risk group (categorical)
- ER/PR/HER2 status (categorical)
- Treatment variables

In [12]:
# Part 3: Univariate Cox Proportional Hazards Analysis (FIXED)
print("="*70)
print("PART 3: UNIVARIATE COX PROPORTIONAL HAZARDS ANALYSIS")
print("="*70)

# Initialize Cox model
cph = CoxPHFitter()

# Storage for hazard ratios
cox_results = []

# Prepare survival data
survival_data = train[['os_days', 'os_status']].copy()

# Continuous variables
print("\n1. CONTINUOUS VARIABLES")
print("="*70)

continuous_vars = ['age', 'stage_imputed', 'lymph_nodes_imputed']

for var in continuous_vars:
    # Prepare data
    data = survival_data.copy()
    data[var] = train[var]
    data = data.dropna()
    
    # Fit Cox model
    cph.fit(data, duration_col='os_days', event_col='os_status')
    
    # Extract results (FIXED)
    summary = cph.summary
    hr = np.exp(summary.loc[var, 'coef'])
    ci_lower = np.exp(summary.loc[var, 'coef lower 95%'])
    ci_upper = np.exp(summary.loc[var, 'coef upper 95%'])
    p_value = summary.loc[var, 'p']
    c_index = cph.concordance_index_
    
    cox_results.append({
        'Variable': var,
        'Type': 'Continuous',
        'Reference': '-',
        'HR': hr,
        'CI_Lower': ci_lower,
        'CI_Upper': ci_upper,
        'P_Value': p_value,
        'C_Index': c_index
    })
    
    print(f"\n{var}:")
    print(f"  HR = {hr:.3f} (95% CI: {ci_lower:.3f}-{ci_upper:.3f})")
    print(f"  p-value = {p_value:.4f}")
    print(f"  C-index = {c_index:.3f}")
    
    if p_value < 0.001:
        print(f"  Significance: *** (p < 0.001)")
    elif p_value < 0.01:
        print(f"  Significance: ** (p < 0.01)")
    elif p_value < 0.05:
        print(f"  Significance: * (p < 0.05)")
    else:
        print(f"  Significance: NS (p >= 0.05)")

# Categorical variables
print("\n" + "="*70)
print("2. CATEGORICAL VARIABLES")
print("="*70)

# Molecular subtype (reference: Hormone Positive)
print("\nMolecular Subtype (ref: Hormone Positive):")
data = survival_data.copy()
data = data.join(pd.get_dummies(train['molecular_subtype'], prefix='mol'))
data = data.dropna()

if 'mol_Hormone_Positive' in data.columns:
    data = data.drop(columns=['mol_Hormone_Positive'])  # Reference category

cph.fit(data, duration_col='os_days', event_col='os_status')
summary = cph.summary

for col in summary.index:
    if col.startswith('mol_'):
        hr = np.exp(summary.loc[col, 'coef'])
        ci_lower = np.exp(summary.loc[col, 'coef lower 95%'])
        ci_upper = np.exp(summary.loc[col, 'coef upper 95%'])
        p_value = summary.loc[col, 'p']
        
        subtype = col.replace('mol_', '').replace('_', ' ')
        
        cox_results.append({
            'Variable': f'Molecular Subtype - {subtype}',
            'Type': 'Categorical',
            'Reference': 'Hormone Positive',
            'HR': hr,
            'CI_Lower': ci_lower,
            'CI_Upper': ci_upper,
            'P_Value': p_value,
            'C_Index': cph.concordance_index_
        })
        
        print(f"  {subtype}: HR = {hr:.3f} (95% CI: {ci_lower:.3f}-{ci_upper:.3f}), p = {p_value:.4f}")

# Risk group (reference: Low Risk)
print("\nRisk Group (ref: Low Risk):")
data = survival_data.copy()
data = data.join(pd.get_dummies(train['risk_group'], prefix='risk'))
data = data.dropna()

if 'risk_Low_Risk' in data.columns:
    data = data.drop(columns=['risk_Low_Risk'])  # Reference category

cph.fit(data, duration_col='os_days', event_col='os_status')
summary = cph.summary

for col in summary.index:
    if col.startswith('risk_'):
        hr = np.exp(summary.loc[col, 'coef'])
        ci_lower = np.exp(summary.loc[col, 'coef lower 95%'])
        ci_upper = np.exp(summary.loc[col, 'coef upper 95%'])
        p_value = summary.loc[col, 'p']
        
        risk = col.replace('risk_', '').replace('_', ' ')
        
        cox_results.append({
            'Variable': f'Risk Group - {risk}',
            'Type': 'Categorical',
            'Reference': 'Low Risk',
            'HR': hr,
            'CI_Lower': ci_lower,
            'CI_Upper': ci_upper,
            'P_Value': p_value,
            'C_Index': cph.concordance_index_
        })
        
        print(f"  {risk}: HR = {hr:.3f} (95% CI: {ci_lower:.3f}-{ci_upper:.3f}), p = {p_value:.4f}")

# ER Status (reference: Negative)
print("\nER Status (ref: Negative):")
data = survival_data.copy()
data = data.join(pd.get_dummies(train['er_status'], prefix='er'))
data = data.dropna()

if 'er_Negative' in data.columns:
    data = data.drop(columns=['er_Negative'])  # Reference category

cph.fit(data, duration_col='os_days', event_col='os_status')
summary = cph.summary

for col in summary.index:
    if col.startswith('er_'):
        hr = np.exp(summary.loc[col, 'coef'])
        ci_lower = np.exp(summary.loc[col, 'coef lower 95%'])
        ci_upper = np.exp(summary.loc[col, 'coef upper 95%'])
        p_value = summary.loc[col, 'p']
        
        status = col.replace('er_', '')
        
        cox_results.append({
            'Variable': f'ER Status - {status}',
            'Type': 'Categorical',
            'Reference': 'Negative',
            'HR': hr,
            'CI_Lower': ci_lower,
            'CI_Upper': ci_upper,
            'P_Value': p_value,
            'C_Index': cph.concordance_index_
        })
        
        print(f"  {status}: HR = {hr:.3f} (95% CI: {ci_lower:.3f}-{ci_upper:.3f}), p = {p_value:.4f}")

# Save Cox results
print("\n" + "="*70)
print("SAVING COX RESULTS")
print("="*70)

cox_df = pd.DataFrame(cox_results)
cox_path = tables_dir / 'univariate_cox_hazard_ratios.csv'
cox_df.to_csv(cox_path, index=False)
print(f"\n✅ Saved: {cox_path}")

print("\nCox regression summary (top 10):")
print(cox_df[['Variable', 'HR', 'CI_Lower', 'CI_Upper', 'P_Value']].head(10).to_string(index=False))

print("\n" + "="*70)
print("PART 3 COMPLETE")
print("="*70)
print(f"\n✅ Analyzed {len(cox_results)} features/groups")
print(f"✅ Hazard ratios calculated and saved")

PART 3: UNIVARIATE COX PROPORTIONAL HAZARDS ANALYSIS

1. CONTINUOUS VARIABLES

age:
  HR = 1.036 (95% CI: 1.030-1.043)
  p-value = 0.0000
  C-index = 0.592
  Significance: *** (p < 0.001)

stage_imputed:
  HR = 1.629 (95% CI: 1.459-1.818)
  p-value = 0.0000
  C-index = 0.588
  Significance: *** (p < 0.001)

lymph_nodes_imputed:
  HR = 1.028 (95% CI: 1.018-1.037)
  p-value = 0.0000
  C-index = 0.582
  Significance: *** (p < 0.001)

2. CATEGORICAL VARIABLES

Molecular Subtype (ref: Hormone Positive):
  HER2 Positive: HR = 1.508 (95% CI: 1.236-1.841), p = 0.0001
  Triple Negative: HR = 1.364 (95% CI: 1.099-1.694), p = 0.0049
  Unknown: HR = 1.249 (95% CI: 0.937-1.665), p = 0.1289

Risk Group (ref: Low Risk):
  High Risk: HR = 2.252 (95% CI: 1.386-3.658), p = 0.0010
  Intermediate Risk: HR = 1.534 (95% CI: 0.944-2.494), p = 0.0841

ER Status (ref: Negative):
  Positive: HR = 0.744 (95% CI: 0.629-0.881), p = 0.0006
  Unknown: HR = 1.836 (95% CI: 0.936-3.602), p = 0.0774

SAVING COX RESULTS


### Part 4: Forest Plot & Multivariate Cox Model

**Objectives:**
1. Create forest plot visualizing all hazard ratios
2. Build multivariate Cox model (adjust for confounders)
3. Compare univariate vs multivariate HRs

In [13]:
# Part 4: Forest Plot & Multivariate Analysis
print("="*70)
print("PART 4: FOREST PLOT & MULTIVARIATE COX ANALYSIS")
print("="*70)

# Create Forest Plot
print("\n1. CREATING FOREST PLOT")
print("="*70)

fig, ax = plt.subplots(figsize=(10, 8))

# Prepare data for forest plot (select key variables)
forest_data = cox_df[~cox_df['Variable'].str.contains('Unknown')].copy()
forest_data = forest_data.sort_values('HR', ascending=True)

y_positions = range(len(forest_data))

# Plot hazard ratios
ax.scatter(forest_data['HR'], y_positions, s=100, color='steelblue', zorder=3)

# Plot confidence intervals
for i, (idx, row) in enumerate(forest_data.iterrows()):
    ax.plot([row['CI_Lower'], row['CI_Upper']], [i, i], 
            color='steelblue', linewidth=2, zorder=2)

# Reference line at HR=1
ax.axvline(x=1, color='red', linestyle='--', linewidth=1.5, label='HR = 1 (No effect)')

# Labels
ax.set_yticks(y_positions)
ax.set_yticklabels(forest_data['Variable'], fontsize=10)
ax.set_xlabel('Hazard Ratio (95% CI)', fontsize=12)
ax.set_title('Forest Plot: Univariate Hazard Ratios for Overall Survival', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlim(0, max(forest_data['CI_Upper']) * 1.1)
ax.grid(alpha=0.3, axis='x')
ax.legend(fontsize=10)

plt.tight_layout()
forest_path = figures_dir / 'forest_plot_univariate.png'
plt.savefig(forest_path)
print(f"✅ Saved: {forest_path}")
plt.close()

# Multivariate Cox Model
print("\n" + "="*70)
print("2. MULTIVARIATE COX MODEL")
print("="*70)

print("\nBuilding multivariate model with key clinical features...")

# Prepare data
multi_data = train[['os_days', 'os_status', 'age', 'stage_imputed', 
                     'lymph_nodes_imputed', 'er_status', 'molecular_subtype']].copy()

# One-hot encode categorical
multi_data = multi_data.join(pd.get_dummies(multi_data['er_status'], prefix='er'))
multi_data = multi_data.join(pd.get_dummies(multi_data['molecular_subtype'], prefix='mol'))

# Drop original categorical and reference categories
multi_data = multi_data.drop(columns=['er_status', 'molecular_subtype'])
if 'er_Negative' in multi_data.columns:
    multi_data = multi_data.drop(columns=['er_Negative'])
if 'mol_Hormone_Positive' in multi_data.columns:
    multi_data = multi_data.drop(columns=['mol_Hormone_Positive'])

# Remove rows with missing
multi_data = multi_data.dropna()

print(f"\nSamples in multivariate model: {len(multi_data)}")

# Fit multivariate Cox
cph_multi = CoxPHFitter()
cph_multi.fit(multi_data, duration_col='os_days', event_col='os_status')

print(f"\nMultivariate model C-index: {cph_multi.concordance_index_:.3f}")

print("\nMultivariate Hazard Ratios:")
summary_multi = cph_multi.summary

# Create clean output
multi_results = []
for var in summary_multi.index:
    hr = np.exp(summary_multi.loc[var, 'coef'])
    ci_lower = np.exp(summary_multi.loc[var, 'coef lower 95%'])
    ci_upper = np.exp(summary_multi.loc[var, 'coef upper 95%'])
    p_value = summary_multi.loc[var, 'p']
    
    # Clean variable name
    clean_var = var
    if var.startswith('er_'):
        clean_var = f"ER {var.replace('er_', '')}"
    elif var.startswith('mol_'):
        clean_var = f"Molecular {var.replace('mol_', '').replace('_', ' ')}"
    
    multi_results.append({
        'Variable': clean_var,
        'HR_Multivariate': hr,
        'CI_Lower_Multi': ci_lower,
        'CI_Upper_Multi': ci_upper,
        'P_Value_Multi': p_value
    })
    
    sig = ""
    if p_value < 0.001:
        sig = "***"
    elif p_value < 0.01:
        sig = "**"
    elif p_value < 0.05:
        sig = "*"
    
    print(f"  {clean_var:30} HR = {hr:.3f} (95% CI: {ci_lower:.3f}-{ci_upper:.3f}), p = {p_value:.4f} {sig}")

# Save multivariate results
multi_df = pd.DataFrame(multi_results)
multi_path = tables_dir / 'multivariate_cox_results.csv'
multi_df.to_csv(multi_path, index=False)
print(f"\n✅ Saved: {multi_path}")

print("\n" + "="*70)
print("PART 4 COMPLETE")
print("="*70)
print(f"\n✅ Forest plot created")
print(f"✅ Multivariate model fitted (C-index: {cph_multi.concordance_index_:.3f})")
print(f"✅ Results saved")

PART 4: FOREST PLOT & MULTIVARIATE COX ANALYSIS

1. CREATING FOREST PLOT
✅ Saved: D:\Projects\tcga-metabric-treatment-ai\results\figures\survival_analysis\forest_plot_univariate.png

2. MULTIVARIATE COX MODEL

Building multivariate model with key clinical features...

Samples in multivariate model: 1995

Multivariate model C-index: 0.659

Multivariate Hazard Ratios:
  age                            HR = 1.041 (95% CI: 1.034-1.047), p = 0.0000 ***
  stage_imputed                  HR = 1.407 (95% CI: 1.251-1.584), p = 0.0000 ***
  lymph_nodes_imputed            HR = 1.024 (95% CI: 1.013-1.036), p = 0.0000 ***
  ER Positive                    HR = 0.697 (95% CI: 0.512-0.949), p = 0.0219 *
  ER Unknown                     HR = 1.836 (95% CI: 0.868-3.885), p = 0.1120 
  Molecular HER2 Positive        HR = 1.431 (95% CI: 1.112-1.840), p = 0.0053 **
  Molecular Triple Negative      HR = 1.198 (95% CI: 0.825-1.740), p = 0.3415 
  Molecular Unknown              HR = 0.873 (95% CI: 0.617-1.236),

## ✓ Exploratory Survival Analysis Complete!

**What we accomplished:**
1. ✅ 5 Kaplan-Meier curves (17 groups analyzed)
2. ✅ Univariate Cox models (10 features)
3. ✅ Forest plot visualization
4. ✅ Multivariate Cox model (C-index: 0.659)

**Key findings:**
- High risk group: 2.25× risk vs Low
- Stage: 63% increased risk per stage
- Age: 4.1% increased risk per year (adjusted)
- ER+: 30% reduced risk (protective)
- HER2+: 43% increased risk (adjusted)

**Files created:**
- 6 KM curves
- 1 forest plot
- 2 hazard ratio tables

**Next:** Pathway-survival associations

In [14]:
# Final Summary
print("="*70)
print("🎉 EXPLORATORY SURVIVAL ANALYSIS COMPLETE!")
print("="*70)

# Count deliverables
import os

km_figures = len([f for f in os.listdir(figures_dir) if f.startswith('km_')])
total_figures = len([f for f in os.listdir(figures_dir) if f.endswith('.png')])
total_tables = len([f for f in os.listdir(tables_dir) if f.endswith('.csv')])

print(f"\n📊 DELIVERABLES:")
print(f"   Figures created: {total_figures}")
print(f"     - KM curves: {km_figures}")
print(f"     - Forest plot: 1")
print(f"   Tables created: {total_tables}")

print(f"\n📈 KEY RESULTS:")
print(f"   Univariate models analyzed: 10")
print(f"   Groups in KM analysis: 17")
print(f"   Multivariate C-index: 0.659")

print(f"\n🔬 SIGNIFICANT PREDICTORS (Multivariate):")
print(f"   Age: HR = 1.041 (p < 0.001) ***")
print(f"   Stage: HR = 1.407 (p < 0.001) ***")
print(f"   Lymph nodes: HR = 1.024 (p < 0.001) ***")
print(f"   ER Positive: HR = 0.697 (p = 0.022) * (PROTECTIVE)")
print(f"   HER2 Positive: HR = 1.431 (p = 0.005) **")

print(f"\n💡 INTERESTING FINDING:")
print(f"   Triple Negative: Significant univariate (p=0.005)")
print(f"                    Not significant multivariate (p=0.342)")
print(f"   → Effect likely mediated by ER status and other factors")

print(f"\n📁 ALL FILES SAVED TO:")
print(f"   Figures: {figures_dir}")
print(f"   Tables: {tables_dir}")

print("\n" + "="*70)
print("✅ SESSION COMPLETE - READY FOR NEXT ANALYSIS")
print("="*70)

# Calculate approximate time for this session
print(f"\n⏱️  ESTIMATED TIME: ~8-10 hours")
print(f"   (KM analysis: 3-4h, Cox models: 3-4h, Multivariate: 2h)")

🎉 EXPLORATORY SURVIVAL ANALYSIS COMPLETE!

📊 DELIVERABLES:
   Figures created: 6
     - KM curves: 5
     - Forest plot: 1
   Tables created: 3

📈 KEY RESULTS:
   Univariate models analyzed: 10
   Groups in KM analysis: 17
   Multivariate C-index: 0.659

🔬 SIGNIFICANT PREDICTORS (Multivariate):
   Age: HR = 1.041 (p < 0.001) ***
   Stage: HR = 1.407 (p < 0.001) ***
   Lymph nodes: HR = 1.024 (p < 0.001) ***
   ER Positive: HR = 0.697 (p = 0.022) * (PROTECTIVE)
   HER2 Positive: HR = 1.431 (p = 0.005) **

💡 INTERESTING FINDING:
   Triple Negative: Significant univariate (p=0.005)
                    Not significant multivariate (p=0.342)
   → Effect likely mediated by ER status and other factors

📁 ALL FILES SAVED TO:
   Figures: D:\Projects\tcga-metabric-treatment-ai\results\figures\survival_analysis
   Tables: D:\Projects\tcga-metabric-treatment-ai\results\tables\survival_analysis

✅ SESSION COMPLETE - READY FOR NEXT ANALYSIS

⏱️  ESTIMATED TIME: ~8-10 hours
   (KM analysis: 3-4h, Cox